In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision #computer vision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize => (-1,1)

transform = transforms.Compose([
    transforms.ToTensor(), # converts into tensors and scales the data
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # standard value so that CIFAR data items gets scaled in (-1,1)
])


trainset = CIFAR10(root="./data", train= True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [01:11<00:00, 2.39MB/s]


In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle = True)
testloader = DataLoader(testset, batch_size=64)

# Building CNN

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32 , kernel_size = 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel_size = 2, stride = 2

            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self , x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening step
        x = self.fc_layers(x)
        return x
        

In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [20]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss function

        loss.backward() # back propagation
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 1.3476251006278845
epoch = 2/10 & loss = 0.9303141937536352
epoch = 3/10 & loss = 0.7397206102109626
epoch = 4/10 & loss = 0.6061386134847045
epoch = 5/10 & loss = 0.49933634490689355
epoch = 6/10 & loss = 0.4027573587491994
epoch = 7/10 & loss = 0.31944194573270696
epoch = 8/10 & loss = 0.23779146901100798
epoch = 9/10 & loss = 0.17543121847940985
epoch = 10/10 & loss = 0.145218712311057


In [21]:
# evaluation

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs , 1) # index with hishest probab

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"'accuracy' = {correct_labels / total_labels * 100}")
    

'accuracy' = 75.01
